In [ ]:
!pip install numpy==1.26.4 pandas==2.2.2 pyarrow==15.0.2 "datasets==2.20.0" --force-reinstall --quiet

In [ ]:
# Static validation: check the relative magnitudes of the score-matching loss and the
# physics term on a synthetic batch BEFORE committing to a full Kaggle training run.
# This avoids the lambda-scaling failure mode (Hypothesis 3).
import torch

torch.manual_seed(0)
B, C, F, T = 4, 1, 256, 256                # SGMSE+ STFT shape with n_fft=510, num_frames=256
sigma_val = 0.5                            # representative diffusion noise level

# Synthesize a smooth clean spectrogram + noisy x_t
freq = torch.linspace(0, 1, F).view(1, 1, F, 1)
time = torch.linspace(0, 1, T).view(1, 1, 1, T)
x_clean = (torch.exp(-freq * 4.0) * torch.cos(2 * 3.14159 * time * 3)).expand(B, C, F, T)
x_clean = torch.complex(x_clean, 0.5 * x_clean.roll(1, dims=-1))
z       = torch.randn_like(x_clean.real) + 1j * torch.randn_like(x_clean.real)
x_t     = x_clean + sigma_val * z
# A perturbed "score" estimate, of the same scale a trained model would produce.
score   = -z / sigma_val + 0.1 * (torch.randn_like(z.real) + 1j * torch.randn_like(z.real))

# (1) Standard score-matching loss: mean over batch of 0.5 * sum |score*sigma + z|^2
sm = torch.square(torch.abs(score * sigma_val + z))
loss_sm = torch.mean(0.5 * torch.sum(sm.reshape(B, -1), dim=-1))

# (2) Physics term: spectral envelope smoothness on log-magnitude of Tweedie x_0_hat
x_hat_spec = x_t + (sigma_val ** 2) * score
mag = torch.abs(x_hat_spec).clamp(min=1e-7)
log_mag = torch.log(mag)
d2_f = log_mag[:, :, 2:, :] - 2.0 * log_mag[:, :, 1:-1, :] + log_mag[:, :, :-2, :]
loss_phys = torch.mean(d2_f ** 2)

print(f'score-matching loss : {loss_sm.item():.4e}')
print(f'physics loss (raw)  : {loss_phys.item():.4e}')
print(f'ratio  phys / sm    : {(loss_phys.item() / loss_sm.item()):.4e}')

# Choose physics_weight so the physics contribution is ~1-5% of the SM loss at init.
target_fraction = 0.02
suggested_w = target_fraction * loss_sm.item() / loss_phys.item()
print(f'suggested physics_weight (for ~{target_fraction*100:.0f}% contribution): {suggested_w:.4e}')

# NaN / inf sanity check on the finite difference path.
assert torch.isfinite(loss_phys), 'physics loss is non-finite!'
assert torch.isfinite(loss_sm),   'score-matching loss is non-finite!'
print('OK: both loss components are finite.')

In [ ]:
# Clone sgmse. Pilot does NOT download the pretrained checkpoint — we train from random init.
import os, shutil
if os.path.exists('/kaggle/working/sgmse'):
    shutil.rmtree('/kaggle/working/sgmse')
os.chdir('/kaggle/working')
get_ipython().system('git clone https://github.com/sp-uhh/sgmse.git')
os.chdir('/kaggle/working/sgmse')
get_ipython().system('pip install -r requirements.txt --quiet')
get_ipython().system('pip install pesq pystoi pandas gdown --quiet')

In [ ]:
# Patch model.py: add spectral-envelope smoothness term inside the score_matching branch.
# We compute a Tweedie estimate of x_0 from the score, then penalize the second
# difference of log|x_0_hat| along the frequency axis.
patch = '''
            # === physics-informed regularizer (spectral envelope smoothness) ===
            # Tweedie estimate of the clean spectrogram from the score.
            # For OUVE-SDE this is an approximation that ignores the drift term;
            # used here as a regularization target, not an exact reconstruction.
            x_hat_spec = x_t + (sigma ** 2) * score
            mag = torch.abs(x_hat_spec).clamp(min=1e-7)
            log_mag = torch.log(mag)
            # Second difference along the frequency axis (axis=2 of (B,C,F,T)).
            d2_f = log_mag[:, :, 2:, :] - 2.0 * log_mag[:, :, 1:-1, :] + log_mag[:, :, :-2, :]
            phys_loss = torch.mean(d2_f ** 2)
            loss = loss + self.physics_weight * phys_loss
'''

with open('/kaggle/working/sgmse/sgmse/model.py', 'r') as f:
    content = f.read()

# Insert physics_weight attribute in __init__ (after self.loss_type = loss_type).
content = content.replace(
    'self.loss_type = loss_type\n',
    'self.loss_type = loss_type\n        self.physics_weight = 0.0\n',
    1,
)

# Insert the physics term at the end of the score_matching branch — right before the
# `elif self.loss_type == "denoiser":` line.
old = '            loss = torch.mean(0.5*torch.sum(losses.reshape(losses.shape[0], -1), dim=-1))\n        elif self.loss_type == "denoiser":'
new = '            loss = torch.mean(0.5*torch.sum(losses.reshape(losses.shape[0], -1), dim=-1))\n' + patch + '        elif self.loss_type == "denoiser":'
assert old in content, 'Anchor for score_matching patch not found'
content = content.replace(old, new, 1)

with open('/kaggle/working/sgmse/sgmse/model.py', 'w') as f:
    f.write(content)

get_ipython().system('grep -n "physics_weight\|phys_loss\|x_hat_spec" /kaggle/working/sgmse/sgmse/model.py')

In [ ]:
import soundfile as sf
import numpy as np
from datasets import load_dataset, Audio

# Step 2 experimental: 3000 train / 100 valid / full 826 test (same as control,
# so the two runs are directly comparable on the same test set).
TEST_DIR = "data/test"
TRAIN_DIR = "data/train"
VALID_DIR = "data/valid"
for d in (TEST_DIR, TRAIN_DIR, VALID_DIR):
    os.makedirs(f"{d}/clean", exist_ok=True)
    os.makedirs(f"{d}/noisy", exist_ok=True)

print("Loading full test set...")
test_data = load_dataset("MeiWu1123/VoiceBank-DEMAND-16k", split="test")
test_data = test_data.cast_column("clean", Audio(sampling_rate=16000))
test_data = test_data.cast_column("noisy", Audio(sampling_rate=16000))
for i, sample in enumerate(test_data):
    clean = np.array(sample["clean"]["array"], dtype=np.float32)
    noisy = np.array(sample["noisy"]["array"], dtype=np.float32)
    fname = f"{i:04d}.wav"
    sf.write(f"{TEST_DIR}/clean/{fname}", clean, 16000)
    sf.write(f"{TEST_DIR}/noisy/{fname}", noisy, 16000)
print(f"Wrote {len(test_data)} test samples")

print("Loading train set (3000 samples)...")
train_data = load_dataset("MeiWu1123/VoiceBank-DEMAND-16k", split="train")
train_data = train_data.cast_column("clean", Audio(sampling_rate=16000))
train_data = train_data.cast_column("noisy", Audio(sampling_rate=16000))
for i in range(3000):
    sample = train_data[i]
    clean = np.array(sample["clean"]["array"], dtype=np.float32)
    noisy = np.array(sample["noisy"]["array"], dtype=np.float32)
    fname = f"{i:04d}.wav"
    sf.write(f"{TRAIN_DIR}/clean/{fname}", clean, 16000)
    sf.write(f"{TRAIN_DIR}/noisy/{fname}", noisy, 16000)
print("Wrote 3000 train samples")

print("Writing validation set (100 samples)...")
for i in range(3000, 3100):
    sample = train_data[i]
    clean = np.array(sample["clean"]["array"], dtype=np.float32)
    noisy = np.array(sample["noisy"]["array"], dtype=np.float32)
    fname = f"{i:04d}.wav"
    sf.write(f"{VALID_DIR}/clean/{fname}", clean, 16000)
    sf.write(f"{VALID_DIR}/noisy/{fname}", noisy, 16000)
print("Wrote 100 validation samples")

In [ ]:
# Step 2 EXPERIMENTAL training: random-init NCSNpp, MSE + physics-informed loss.
# Identical to control_from_scratch.ipynb except:
#   1. SAVE_DIR = sgmse_experimental
#   2. physics_weight is CALIBRATED at runtime to give ~2% gradient contribution
#      on a real random-init-model batch (the synthetic check at the top of the
#      notebook is just a sanity check; real-batch magnitudes can differ a lot)

finetune_script = '''
import os
os.chdir("/kaggle/working/sgmse")
import torch
if not hasattr(torch, "_load_patched"):
    _orig = torch.load
    def _patched(*a, **kw):
        kw["weights_only"] = False
        return _orig(*a, **kw)
    torch.load = _patched
    torch._load_patched = True

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, Callback
from sgmse.model import ScoreModel
from sgmse.data_module import SpecsDataModule

SAVE_DIR  = "/kaggle/working/sgmse_experimental"
DATA_DIR  = "/kaggle/working/sgmse/data"
os.makedirs(SAVE_DIR, exist_ok=True)

pl.seed_everything(42)

model = ScoreModel(
    backbone="ncsnpp",
    sde="ouve",
    data_module_cls=SpecsDataModule,
    theta=1.5, sigma_min=0.05, sigma_max=0.5, N=1000,
    loss_type="score_matching", loss_weighting="sigma^2",
    num_eval_files=0, lr=1e-4,
    nf=32,
    base_dir=DATA_DIR, format="default", batch_size=16,
    n_fft=510, hop_length=128, num_frames=256, window="hann",
    num_workers=2, dummy=False, spec_factor=0.15, spec_abs_exponent=0.5,
    normalize="noisy", transform_type="exponent",
)

# === CALIBRATE physics_weight ===
# Get a real batch and run two forward passes with the same RNG state but
# different physics_weight values; subtract to recover loss_phys.
model.cuda()
model.eval()
model.data_module.setup(stage="fit")
batch = next(iter(model.data_module.train_dataloader()))
batch = [b.cuda() if torch.is_tensor(b) else b for b in batch]

def _compute_loss(seed):
    """Call whatever internal loss method this sgmse version exposes."""
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(model, "_step"):
        return float(model._step(batch, 0))
    # Fall back to training_step, but mute self.log (errors outside trainer).
    orig_log = model.log
    model.log = lambda *a, **kw: None
    try:
        return float(model.training_step(batch, 0))
    finally:
        model.log = orig_log

with torch.no_grad():
    model.physics_weight = 0.0
    loss_sm = _compute_loss(123)
    model.physics_weight = 1.0
    loss_total = _compute_loss(123)

loss_phys = loss_total - loss_sm
print(f"[calibration] loss_sm   : {loss_sm:.4e}")
print(f"[calibration] loss_phys : {loss_phys:.4e}")

target_fraction = 0.02   # physics contributes ~2% of total gradient
calibrated_w = target_fraction * loss_sm / max(abs(loss_phys), 1e-12)
print(f"[calibration] target_fraction = {target_fraction}")
print(f"[calibration] CALIBRATED physics_weight = {calibrated_w:.4e}")

model.physics_weight = calibrated_w
# trainer.fit will set training mode on its own; skip model.train()
# (this sgmse subclass overrides train(mode) with no default).
# === end calibration ===

class PrintLosses(Callback):
    def on_validation_epoch_end(self, trainer, pl_module):
        m = trainer.callback_metrics
        vl = m.get("valid_loss"); tl = m.get("train_loss_epoch")
        print("\\n[Epoch " + str(trainer.current_epoch) +
              "] train_loss=" + (str(round(float(tl),4)) if tl is not None else "?") +
              " | valid_loss=" + (str(round(float(vl),4)) if vl is not None else "?") +
              " | physics_weight=" + str(pl_module.physics_weight) + "\\n")

ckpt_cb = ModelCheckpoint(
    dirpath=SAVE_DIR,
    filename="experimental_epoch{epoch:02d}_valloss{valid_loss:.4f}",
    save_top_k=3, monitor="valid_loss", mode="min", every_n_epochs=1,
    save_last=True,
)

trainer = pl.Trainer(
    max_epochs=40, accelerator="gpu", devices=1,
    callbacks=[ckpt_cb, PrintLosses()],
    log_every_n_steps=20, enable_progress_bar=True,
    gradient_clip_val=1.0,
)
trainer.fit(model)
print("Best checkpoint:", ckpt_cb.best_model_path)
'''

with open('/kaggle/working/sgmse/finetune_experimental.py', 'w') as f:
    f.write(finetune_script)
print('finetune_experimental.py written')

In [ ]:
# Run Step 2 experimental. Calibration prints loss_sm / loss_phys / calibrated
# physics_weight at the top of training output — verify physics_weight isn't
# negative or absurdly large (e.g., > 1e6) before letting it train to completion.
# Estimated ~5-6 hours on Kaggle T4.
get_ipython().system('python finetune_experimental.py')

In [ ]:
# 1. Patch enhancement.py
with open('/kaggle/working/sgmse/enhancement.py', 'r') as f:
  content = f.read()

patch = '''import torch
_original_torch_load = torch.load
def _patched_torch_load(*args, **kwargs):
  kwargs['weights_only'] = False
  return _original_torch_load(*args, **kwargs)
torch.load = _patched_torch_load

'''
if '_patched_torch_load' not in content:
  with open('/kaggle/working/sgmse/enhancement.py', 'w') as f:
      f.write(patch + content)
  print('enhancement.py patched')
else:
  print('enhancement.py already patched')

# 2. Patch Lightning's loaders (pure Python, no sed)
for path in [
  '/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py',
  '/usr/local/lib/python3.12/dist-packages/lightning_fabric/utilities/cloud_io.py',
]:
  with open(path, 'r') as f:
      src = f.read()
  new = src.replace(
      'weights_only: Optional[bool] = None,',
      'weights_only: Optional[bool] = False,',
  )
  if new != src:
      with open(path, 'w') as f:
          f.write(new)
      print('patched', path)
  else:
      print('no change needed', path)

# 3. Clear any partial output from a previous experimental run
import shutil, os
if os.path.exists('/kaggle/working/sgmse/enhanced_experimental'):
  shutil.rmtree('/kaggle/working/sgmse/enhanced_experimental')
  print('cleared enhanced_experimental/')

In [ ]:
# Enhancement + metrics on the trained experimental checkpoint.
# Same test set / N as control_from_scratch so the two runs are directly comparable.
import glob, os
ckpts = sorted(glob.glob('/kaggle/working/sgmse_experimental/experimental_*.ckpt'))
print('checkpoints:', ckpts)
CKPT = ckpts[-1]
print('using:', CKPT)

get_ipython().system(f'python enhancement.py --test_dir data/test/noisy --enhanced_dir enhanced_experimental --ckpt {CKPT} --N 10')
get_ipython().system('python calc_metrics.py --clean_dir data/test/clean --noisy_dir data/test/noisy --enhanced_dir enhanced_experimental')